<a href="https://colab.research.google.com/github/PauloRadatz/opendss-python-examples/blob/main/presentations/ieee_et_pes_pels_joint_chapter_workshop/04_introduction_to_py_dss_toolkit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4 — Introduction to py-dss-toolkit

In Notebooks 1 to 3 we used only `py-dss-interface`. That package is the bridge between Python and OpenDSS — it lets us send any OpenDSS command from Python and read results back. Everything we did was possible with just that.

But you probably noticed a pattern: a lot of the code we wrote was about *organizing* the results — pairing voltages with node names, building DataFrames, plotting. That work shows up in every study. So I built `py-dss-toolkit` to handle the common pieces.

Think of it this way: **`py-dss-interface` is how Python talks to OpenDSS. `py-dss-toolkit` builds on top of it to make common workflows shorter, cleaner, and more visual.**

In this notebook we redo parts of the previous workflows using the toolkit and compare the result.

## Setup

Installing `py-dss-toolkit` automatically pulls `py-dss-interface` as a dependency.

In [ ]:
!pip install py-dss-toolkit
!git clone https://github.com/PauloRadatz/opendss-python-examples.git
%cd opendss-python-examples

In [ ]:
import pathlib
import py_dss_interface
from py_dss_toolkit import dss_tools

start_dir = pathlib.Path.cwd().resolve()
repo_root = next(
    parent for parent in [start_dir, *start_dir.parents]
    if (parent / "feeder_models").exists()
)
dss_file = repo_root / "feeder_models" / "EPRITestCircuits" / "ckt5" / "Master_ckt5.dss"

## Link the DSS instance to the toolkit

We still create a normal `py_dss_interface.DSS()` object — that is unchanged. We then connect it to `dss_tools` with `update_dss(dss)`. From now on we have two ways to control OpenDSS: directly through `dss`, or through the higher-level helpers in `dss_tools`.

In [ ]:
dss = py_dss_interface.DSS()
dss_tools.update_dss(dss)

print(f"OpenDSS started: {dss.started}")

## Compile and solve

Same as before. The toolkit does not replace the OpenDSS engine; it sits on top of it.

In [ ]:
dss.text(f"compile [{dss_file}]")
dss.text("solve")

## Access the model as DataFrames

`dss_tools.model` exposes structured DataFrames for the elements in the circuit. No more manual loops to collect properties — the toolkit does it for us.

Below we look at lines, buses, and a model summary. The `buses` DataFrame in particular is hard to assemble in standalone OpenDSS or with `py-dss-interface` alone — every bus, every property, in one table.

In [ ]:
lines_df = dss_tools.model.lines_df
print(f"Lines in the circuit: {len(lines_df)}")
lines_df.head()

In [ ]:
buses_df = dss_tools.model.buses_df
print(f"Buses in the circuit: {len(buses_df)}")
buses_df.head()

`dss_tools.model.summary_df` gives a quick model summary — number of buses, lines, transformers, loads, and so on. Useful as a sanity check after compiling.

In [ ]:
model_summary = dss_tools.model.summary_df
model_summary

## Get snapshot results in one line

`dss_tools.results.summary_df` returns the most useful snapshot quantities — feederhead P and Q, total losses, min and max voltage — already organized in a small table.

In [ ]:
results_summary = dss_tools.results.summary_df
results_summary

## Find the lowest voltage bus — toolkit version

Compare this with what we wrote in Notebook 3. The work is the same, but we are now reading an already-organized DataFrame.

`dss_tools.results.voltage_ln_nodes` returns a tuple of two DataFrames — magnitudes (in pu) and angles (in degrees). Each row is a bus, and the columns are `node1`, `node2`, `node3`.

In [ ]:
vmag_df, vang_df = dss_tools.results.voltage_ln_nodes
vmag_df.head()

In [ ]:
# Stack the per-node columns into a long Series, drop zero values
# (zero means the phase is not present at that bus), and find the minimum.
vmag_long = vmag_df.stack()
vmag_long = vmag_long[vmag_long > 0]

min_idx = vmag_long.idxmin()
min_bus, min_node = min_idx
min_v = vmag_long[min_idx]

print(f"Lowest voltage on the feeder")
print(f"  Bus     : {min_bus}")
print(f"  Node    : {min_node}")
print(f"  Voltage : {min_v:.4f} pu")

## Add an energymeter and re-solve

Voltage profile and circuit plots in `py-dss-toolkit` rely on an energymeter so the toolkit knows where the feederhead is. The next cell uses a small toolkit helper that adds a line in series with the source plus an energymeter and a few monitors. Then we re-solve.

In [ ]:
dss_tools.model.add_line_in_vsource(add_meter=True, add_monitors=True)
dss.text("solve")

## Voltage profile in one line

An interactive voltage profile of the entire feeder, color-coded by phase. This kind of plot is one of the things that took the most code before the toolkit existed.

In [ ]:
dss_tools.interactive_view.voltage_profile(title="ckt5 voltage profile | peak load")

## Circuit map colored by voltage

Same idea, different view — the whole feeder on a map, with voltage as the color. Useful for spotting where the low-voltage area actually sits geographically.

In [ ]:
dss_tools.interactive_view.circuit_plot(
    parameter="voltage",
    title="ckt5 voltage map | peak load",
)

## Side-by-side: with and without the toolkit

Same workflow, two ways to write it.

| Task | py-dss-interface | py-dss-toolkit |
|---|---|---|
| Read all line properties | manual loop over `dss.lines` | `dss_tools.model.lines_df` |
| Read all bus properties | manual loop over `dss.bus` | `dss_tools.model.buses_df` |
| Snapshot summary | manual call to `total_power`, `losses`, `buses_vmag_pu`, `min/max` | `dss_tools.results.summary_df` |
| Lowest voltage node | pair `nodes_names` and `buses_vmag_pu`, find min | `dss_tools.results.voltage_ln_nodes` + `stack()` + `idxmin()` |
| Voltage profile plot | build matplotlib by hand from monitors | `dss_tools.interactive_view.voltage_profile()` |
| Circuit map | not directly available | `dss_tools.interactive_view.circuit_plot()` |

The toolkit is not a replacement. Underneath, it is calling the same `py-dss-interface` we used in Notebooks 1 to 3.

## Key takeaways

- `py-dss-interface` gives you direct, low-level control of OpenDSS from Python. That is the foundation.
- `py-dss-toolkit` builds on top of it with structured DataFrames, ready-made result tables, and interactive visualizations.
- The two work together. You always have `dss.text(...)` available — the toolkit just saves you from rewriting the same boilerplate.

This is the message of the workshop: OpenDSS is a powerful standalone simulator, but with `py-dss-interface` it becomes programmable, and with `py-dss-toolkit` the common engineering workflows become short and visual.

## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [pauloradatz.me](https://www.pauloradatz.me)
- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)

## Contact

For questions or follow-up about these materials:

- Paulo Radatz
- Email: [paulo.radatz@gmail.com](mailto:paulo.radatz@gmail.com)
- LinkedIn: [linkedin.com/in/pauloradatz](https://www.linkedin.com/in/pauloradatz/)
- Website: [pauloradatz.me](https://www.pauloradatz.me/)